In [1]:
!pip3 install ultralytics


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14 -m pip install --upgrade pip


In [10]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')
res = model.predict("image.png")
res[0].show()


image 1/1 /Users/aruvins/projects/AI Projects/computer-Vision/YOLO26_Models/YOLOv26/image.png: 480x640 1 person, 2 dogs, 1 cup, 1 couch, 2 potted plants, 1 dining table, 1 tv, 1 laptop, 1 remote, 1 book, 21.6ms
Speed: 1.4ms preprocess, 21.6ms inference, 0.1ms postprocess per image at shape (1, 3, 480, 640)


In [6]:
model.export(format="onnx")

Ultralytics 8.4.37 🚀 Python-3.14.0 torch-2.11.0 CPU (Apple M4 Max)

PyTorch: starting from 'yolo26n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.3 MB)

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.93...


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/ultralytics/nn/modules/head.py:178: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if self.dynamic or self.shape != shape:
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: export success ✅ 0.9s, saved as 'yolo26n.onnx' (9.5 MB)

Export complete (0.9s)
Results saved to /Users/aruvins/projects/AI Projects/computer-Vision/YOLO26_Models/YOLOv26
Predict:         yolo predict task=detect model=yolo26n.onnx imgsz=640 
Validate:        yolo val task=detect model=yolo26n.onnx imgsz=640 data=/home/lq/codes/ultralytics/ultralytics/cfg/datasets/coco.yaml  
Visualize:       https://netron.app


'yolo26n.onnx'

### Run the model from locally downloaded file

In [8]:
import cv2
from ultralytics import YOLO

# Load the ONNX model
model = YOLO("yolo26n.onnx", task="detect")

# Run inference on an image or video
results = model("image.png")

# Show the results
results[0].show()

Loading yolo26n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.26.0 with CPUExecutionProvider

image 1/1 /Users/aruvins/projects/AI Projects/computer-Vision/YOLO26_Models/YOLOv26/image.png: 640x640 1 person, 1 dog, 1 cup, 1 couch, 2 potted plants, 1 dining table, 1 tv, 1 remote, 21.8ms
Speed: 1.4ms preprocess, 21.8ms inference, 0.1ms postprocess per image at shape (1, 3, 640, 640)


| Aspect | YOLOv8 / YOLOv11 | YOLOv26 |
| :--- | :--- | :--- |
| **NMS Dependency** | Required | Removed (One-to-One Head) |
| **Bounding Box Regression** | DFL-based | Simplified regression |
| **Quantization Robustness** | Moderate | High (INT8/FP16 friendly) |
| **Edge Deployment** | Manual optimization often needed | Clean, end-to-end export |
| **Latency Stability** | Scene-dependent | Deterministic |

### Benefits of YOLOv26

1) Eliminates NMS (Non-Maximum Suppression)
    - Non-Maximum Suppression (NMS) is a post-processing algorithm used in object detection to clean up the output of a neural network.

    - NMS works by throwing away all boxes with a low probability score and then sorts the remaining boxes from lowest to highest confidence. NMS then takes the best box and compares it to all others. If a box has a high IOU (Intersection over Union), it assumes they are describing the same object. It repeats until one box per object remains.

    - YOLOv26 is End to End so it is trained to predict exactly one box per object and NMS is now unnecessary

2) Remove DFL (Distribution Focal Loss)
    - DFL is a specialized way for the neural network to predict boundaries of an object. DFL predicts a probability distribution for where an edge may be

    - YOLOv26 simplified the way boundaries are detected by predicting it directly. This makes boundary detection less accurate but significantly reduces the size of the model

    

        | Feature | DFL (v8 / v11) | Simplified (v26) |
        | :--- | :--- | :--- |
        | **Logic** | Predicts a "cloud" of probabilities for edges. | Predicts coordinates more directly. |
        | **Precision** | Excellent for ambiguous edges. | Very high (optimized via newer training methods). |
        | **Speed** | Slightly slower due to extra math. | Faster and more "hardware-friendly." |
        | **Exporting** | Can be tricky for some formats. | Highly streamlined for ONNX/CoreML. |
    